# SwarmSort Gen 2, notebook 6: final test-set evaluation of Arms A, B and C

**What:** scores the three Gen 2 arms (A: defaults, B: random search, C: PSO) on the held-out test split with
`src/evaluate.py --arms configs/gen2/arms.yaml`: once on the whole test split (tag `all`) and once per domain
(`studio`, `real_world`, and `india` if available). Each run gives mAP@50 and mAP@50-95 with 95% paired-bootstrap
confidence intervals, pairwise differences, per-class AP, confusion matrices, latency and sample predictions.

**Why:** this is the final, fair comparison. **The test split is used exactly once, here**: run this notebook only
after all three arms are trained, and do not tune anything on its numbers.

**Inputs** (Add Input -> Notebook Output, the finished version of each):

| Notebook output | Gives |
|---|---|
| `g2_0_build_dataset` | the test images |
| `g2_1_arm_a` | `gen2_arm_a_best.pt`, `arm_a.json`, `arm_a_epochs.csv` |
| `g2_4_arm_b` | `gen2_arm_b_best.pt`, `arm_b.json`, `arm_b_epochs.csv`, `best_random.json` |
| `g2_5_arm_c` | `gen2_arm_c_best.pt`, `arm_c.json`, `arm_c_epochs.csv`, `best_pso.json` |

Optional, for the India test set (Add Input -> Datasets): **your private DWSD dataset** (see
`notebooks/gen2/README.md`). The g2_0 build was made without DWSD, so this notebook builds a separate, test-only
DWSD dataset with `build_dataset.py --only dwsd` (same config, so the same 7 classes) and scores it with tag `india`.
DWSD is test-only, so the training data does not change. Without DWSD, india is skipped with a message;
if the DWSD build or its evaluation fails, the error is printed and the other domains still run.

The locating cell finds these files anywhere under `/kaggle/input` and copies them to the paths
`configs/gen2/arms.yaml` expects. **Accelerator: GPU T4 x2**, **Internet: on**. Run it with
**Save Version -> Save & Run All (Commit)**.

**Expected time:** about 30-60 minutes (4 evaluation runs; each validates the three models, times them and draws 1000
bootstrap resamples; Gen 1's single run took about 5 minutes for 1046 test images), plus a few minutes to build the
DWSD test set (784 photos) if it is attached.

**Freeze protection.** The attached dataset is first copied to local disk, and every script runs under a watchdog
that restarts it after 20 minutes without output or file changes (at most 3 times).

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/SwarmSort"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Shreyansh303/SwarmSort.git {REPO}
assert os.path.isdir(f"{REPO}/src"), "git clone failed: turn on Settings -> Internet and run again"
%cd {REPO}
# Kaggle's own CUDA build of torch is kept: only Ultralytics is installed, at the version pinned in requirements.txt
!pip install -q ultralytics==8.4.169
import torch, ultralytics
assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator -> GPU T4 x2 and run again"
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))
if not os.path.isfile("src/watchdog_run.py"):
    raise RuntimeError("src/watchdog_run.py is not on GitHub yet: push the latest commits")
sys.path.insert(0, f"{REPO}/src")
from watchdog_run import run_watched
# About 10 s: check on this machine that a frozen script and its worker process are killed and retried
subprocess.run([sys.executable, "-m", "unittest", "-q",
                "test_watchdog.TestWatchdog.test_b_stalled_script_is_killed_and_retried",
                "test_watchdog.TestWatchdog.test_e_child_processes_are_killed_too"], cwd="tests", check=True)


def run(*args, watch=(), stall_minutes=20, max_retries=3):
    """Run a repository script under the stall watchdog (src/watchdog_run.py). Its output is streamed here. If it
    prints nothing and nothing changes under `watch` for stall_minutes, it is killed together with its dataloader
    workers and started again with the same arguments, at most max_retries times (search.py then continues from its
    log, train_final.py --resume from last.pt). If the script stops with an error, this cell fails and the version
    is marked failed (as with check=True)."""
    args = [str(a) for a in args]
    print("$ python", " ".join(args), flush=True)
    run_watched([sys.executable, *args], watch=watch, stall_minutes=stall_minutes, max_retries=max_retries,
                env={**os.environ, "PYTHONUNBUFFERED": "1"})


In [ ]:
# Find the dataset built by g2_0_build_dataset (attached as input), copy it to this machine's local disk, and write
# configs that point at the local copy: /kaggle/working/gen2_cfg/data.yaml, data_test_<domain>.yaml, split.csv, lists/
# Why the copy: two Gen 2 runs froze about 55 minutes into training and sat idle until Kaggle's 12-hour limit,
# most likely because the attached notebook-output mount under /kaggle/input stopped answering. Training now reads
# only the local copy. It goes outside /kaggle/working, so it does not become part of this version's output.
# The copy reads that mount too, so it fails loudly (TimeoutError) if no file arrives for 2 minutes.
import glob, json, shutil
from watchdog_run import copy_files, list_tree
INPUT, CFG = "/kaggle/input", "/kaggle/working/gen2_cfg"
builds = sorted({os.path.dirname(p) for p in glob.glob(f"{INPUT}/**/build_report.json", recursive=True)
                 if os.path.isfile(os.path.join(os.path.dirname(p), "data.yaml"))
                 and os.path.isdir(os.path.join(os.path.dirname(p), "images"))})
if not builds:
    raise FileNotFoundError("The Generation 2 dataset is not attached. Add Input -> Notebook Output -> "
                            "g2_0_build_dataset (its finished version), then run again.")
if len(builds) > 1:
    raise RuntimeError("Several built datasets are attached; remove all but one version of g2_0_build_dataset:\n  "
                       + "\n  ".join(builds))
BUILD_SRC = builds[0]
print("Built dataset (attached):", BUILD_SRC)
files = list_tree(BUILD_SRC, exclude=["*.cache"])
size = sum(n for _, n in files)
for base in ("/kaggle/tmp", "/kaggle/temp", "/tmp"):  # local disk outside /kaggle/working
    if os.path.isdir(base) and os.access(base, os.W_OK) and shutil.disk_usage(base).free > 1.1 * size + 2e9:
        BUILD = f"{base}/gen2_data"
        break
else:
    raise OSError(f"No local folder has {size / 1e9:.1f} GB free for the dataset copy")
stamp = {"source": BUILD_SRC, "files": len(files), "bytes": size}
if os.path.isfile(f"{BUILD}/.copied.json") and json.load(open(f"{BUILD}/.copied.json")) == stamp:
    print("Local copy already made in this session:", BUILD)
else:
    shutil.rmtree(BUILD, ignore_errors=True)
    print(f"Copying {len(files)} files ({size / 1e9:.2f} GB) to {BUILD}", flush=True)
    info = copy_files(BUILD_SRC, BUILD, files, workers=16, stall_seconds=120)
    if list_tree(BUILD) != files:
        raise RuntimeError(f"The local copy {BUILD} differs from {BUILD_SRC} (file names or sizes)")
    json.dump(stamp, open(f"{BUILD}/.copied.json", "w"))
    print(f"Copied {info['files']} files, {info['bytes'] / 1e9:.2f} GB in {info['seconds']:.0f} s "
          f"({info['bytes'] / 1e6 / max(info['seconds'], 1):.0f} MB/s); names and sizes match the attached build")
for sub in ("images", "labels"):
    print(f"  {sub}/: {len(os.listdir(f'{BUILD}/{sub}'))} files")
if "--relocate-from" not in open("src/build_dataset.py", encoding="utf-8").read():
    raise RuntimeError("src/build_dataset.py on GitHub has no --relocate-from mode yet: push the latest commits")
run("src/build_dataset.py", "--relocate-from", BUILD, "--out", CFG)
assert os.path.isfile(f"{CFG}/data.yaml"), f"the relocation did not write {CFG}/data.yaml"
print(open(f"{CFG}/data.yaml").read())
print("Test-set yamls:", sorted(os.path.basename(p) for p in glob.glob(f"{CFG}/data_test_*.yaml")))


In [ ]:
# India (DWSD) test set. The g2_0 build has none if DWSD was not attached there. DWSD is test-only (force_split: test,
# domain india) and needs nothing from the other sources, so if it is attached here it is built on its own:
# build_dataset.py --only dwsd into /kaggle/working/gen2_india, with the same config (so the same 7 classes in the
# same order) and the same --seed and --max-side as the main build. DWSD comes as grey-level semantic masks
# (format: semantic_mask in configs/gen2/sources.yaml): each connected region of a class value becomes one box.
# The duplicate grouping of build_dataset.py is not run between DWSD and the training sources (they are in different
# builds): DWSD is a separate collection (Kolkata streets) and test-only. A cheap MD5 cross-check against every image
# of the main build looks for byte-identical copies instead (both builds re-encode with the same settings, so the
# same photo gives the same file; resized or re-compressed near-copies are not caught).
# India is optional: if anything here fails, the error is printed and the notebook goes on with the other domains.
import glob, hashlib, json, traceback, yaml
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
SOURCES, INDIA_OUT = "configs/gen2/sources.yaml", "/kaggle/working/gen2_india"
main_names = yaml.safe_load(open(f"{CFG}/data.yaml"))["names"]


def india_test_set():
    """Find or build the India test set. Returns (data yaml or None, note for the results table or None)."""
    if os.path.isfile(f"{CFG}/data_test_india.yaml"):
        print("The attached build has its own India test set:", f"{CFG}/data_test_india.yaml")
        return f"{CFG}/data_test_india.yaml", None
    code = open("src/build_dataset.py", encoding="utf-8").read()
    if "--only" not in code or "semantic_mask" not in code:
        raise RuntimeError("src/build_dataset.py on GitHub has no --only option or semantic_mask reader yet: "
                           "push the latest commits")
    sys.path.insert(0, "src")
    import build_dataset as bd
    _, (dwsd,) = bd.load_config(SOURCES, only=["dwsd"])
    wanted = {bd.norm_name(n) for n in dwsd["locate"]}
    found = [d.as_posix() for d in bd.walk_dirs(Path(INPUT), skip=[BUILD_SRC])
             if d != Path(INPUT) and bd.name_matches(d.name, wanted, dwsd["locate_match"])]
    if not found:
        print(f"No India test set: the attached build has none and no DWSD dataset is attached (no folder named "
              f"one of {dwsd['locate']} under {INPUT}). india is skipped. To score it, Add Input -> your private "
              f"DWSD dataset (see notebooks/gen2/README.md) and run again.")
        return None, None
    print("DWSD found:", found[0], f"(+{len(found) - 1} more)" if len(found) > 1 else "")
    settings = json.load(open(f"{BUILD}/build_report.json"))["settings"]
    try:
        run("src/build_dataset.py", "--sources", SOURCES, "--out", INDIA_OUT, "--search-root", INPUT,
            "--seed", settings["seed"], "--max-side", settings["max_side"], "--only", "dwsd")
    except subprocess.CalledProcessError:
        raise RuntimeError("the DWSD-only build failed; its error is printed above (e.g. a mask value missing from "
                           "the dwsd mask_values in configs/gen2/sources.yaml)") from None
    india_yaml = f"{INDIA_OUT}/data_test_india.yaml"
    india_names = yaml.safe_load(open(india_yaml))["names"]
    assert india_names == main_names, f"class names differ: DWSD build {india_names}, main build {main_names}"
    india_report = json.load(open(f"{INDIA_OUT}/build_report.json"))
    n_india = india_report["test_variants"]["india"]["test_images"]
    assert n_india > 0 and india_report["totals"]["train"]["images"] == 0, "DWSD must give test images only"

    def md5(path):
        return hashlib.md5(open(path, "rb").read()).hexdigest()

    main_imgs = sorted(glob.glob(f"{BUILD}/images/*"))
    india_imgs = sorted(glob.glob(f"{INDIA_OUT}/images/*"))
    with ThreadPoolExecutor(8) as pool:
        main_md5, india_md5 = list(pool.map(md5, main_imgs)), list(pool.map(md5, india_imgs))
    by_md5 = defaultdict(list)
    for p, h in zip(main_imgs, main_md5):
        by_md5[h].append(os.path.basename(p))
    same = [{"india": os.path.basename(p), "main_build": by_md5[h]} for p, h in zip(india_imgs, india_md5)
            if h in by_md5]
    india_report["md5_cross_check"] = {"india_images": len(india_imgs), "main_build_images": len(main_imgs),
                                       "identical_files": len(same), "pairs": same}
    os.makedirs("results/gen2", exist_ok=True)
    json.dump(india_report, open("results/gen2/build_report_india.json", "w"), indent=2)
    print(f"India test set: {n_india} DWSD images, the same {len(main_names)} classes as the main build: {india_yaml}")
    print(f"MD5 cross-check: {len(india_imgs)} DWSD images vs {len(main_imgs)} main-build images: "
          f"{len(same)} identical files")
    for pair in same[:20]:
        print("  identical:", pair["india"], "==", ", ".join(pair["main_build"]))
    if same:
        print("WARNING: these DWSD images are also in the main build; see results/gen2/build_report_india.json")
    return india_yaml, (f"india: DWSD-only test set built in g2_6 (build_dataset.py --only dwsd), {n_india} images; "
                        f"MD5 cross-check against the {len(main_imgs)} main-build images: {len(same)} identical files")


try:
    INDIA_YAML, INDIA_NOTE = india_test_set()
except Exception as e:
    traceback.print_exc()
    print(f"\n*** WARNING: the India (DWSD) test set could not be prepared: {type(e).__name__}: {e}\n"
          f"*** india is skipped; the other domains are evaluated as usual. ***")
    INDIA_YAML, INDIA_NOTE = None, f"india: skipped, the DWSD test set could not be prepared ({type(e).__name__}: {e})"


In [ ]:
# Find each arm's weights, training JSON and epochs CSV in the attached outputs, and copy them to the paths in
# configs/gen2/arms.yaml (models at results/gen2/runs/arm_<x>/weights/best.pt, JSONs and CSVs in results/gen2/)
import hashlib, shutil, yaml
from collections import defaultdict
ARMS_YAML = "configs/gen2/arms.yaml"
arms = yaml.safe_load(open(ARMS_YAML))["arms"]
NOTEBOOK = {"A": "g2_1_arm_a", "B": "g2_4_arm_b", "C": "g2_5_arm_c"}
index = defaultdict(list)  # file name -> every path under /kaggle/input (image and label folders skipped)
for root, dirs, files in os.walk(INPUT):
    dirs[:] = [d for d in dirs if d not in ("images", "labels")]
    for f in files:
        index[f].append(os.path.join(root, f).replace(os.sep, "/"))


def sha(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()


def closest(paths, anchor):
    """The path sharing the longest folder prefix with anchor, i.e. from the same attached output."""
    return max(paths, key=lambda p: (len(os.path.commonpath([p, anchor])), p))


used = []
for label, arm in arms.items():
    x = label.lower()
    nb = NOTEBOOK.get(label, "the notebook that trained arm " + label)
    weights = index[f"gen2_arm_{x}_best.pt"] + [p for p in index["best.pt"]
                                                  if p.endswith(f"results/gen2/runs/arm_{x}/weights/best.pt")]
    if not weights:
        raise FileNotFoundError(f"No weights for arm {label} (gen2_arm_{x}_best.pt) under {INPUT}: "
                                f"Add Input -> Notebook Output -> {nb} (its finished version), then run again.")
    if len({sha(p) for p in weights}) > 1:
        raise RuntimeError(f"Different weights for arm {label} are attached (several versions of {nb}?); keep one:\n  "
                           + "\n  ".join(weights))
    anchor = sorted(weights, key=len)[0]  # the top-level copy if present
    os.makedirs(os.path.dirname(arm["model"]), exist_ok=True)
    shutil.copyfile(anchor, arm["model"])
    used.append((label, "model", anchor, arm["model"]))
    for key in ("val_json", "epochs_csv", "config"):
        dest = arm.get(key)
        if not dest or any(u[3] == dest for u in used):
            continue  # arm A's config is its val_json, already copied
        name = os.path.basename(dest)
        found = [p for p in index[name] if p.endswith(f"results/gen2/{name}")]
        committed = os.path.exists(dest)
        if key == "config" and committed:  # a search winner committed to the repository is authoritative
            used.append((label, key, "(repository)", dest))
        elif found:  # the copy from the same output as the weights belongs to them
            src = closest(found, anchor)
            if committed and sha(src) != sha(dest):
                print(f"Note: {dest} differs from the committed file; using {src}, which came with the weights")
            shutil.copyfile(src, dest)
            used.append((label, key, src, dest))
        elif committed:
            used.append((label, key, "(repository)", dest))
        else:
            raise FileNotFoundError(f"{name} for arm {label} not found in the repository or under {INPUT}: attach "
                                    f"the finished version of {nb}, or commit {dest} and push.")
print(f"{'arm':<4}{'file':<11}{'copied to':<42}from")
for label, key, src, dest in used:
    print(f"{label:<4}{key:<11}{dest:<42}{src}")


In [ ]:
# The test split, scored once: the whole test set (tag all), then each domain. evaluate.py checks every checkpoint
# (7 classes, 100 epochs, imgsz 640 and its arm's hyperparameters) before it validates anything.
# india: the test set found or built by the India cell above (skipped if there is none).
DATA = [("all", f"{CFG}/data.yaml")] + [(d, f"{CFG}/data_test_{d}.yaml") for d in ("studio", "real_world")]
for tag, data in DATA:
    if not os.path.exists(data):
        raise FileNotFoundError(f"{data} missing: the build has no {tag} test set (see g2_0_build_dataset)")
if INDIA_YAML:
    DATA.append(("india", INDIA_YAML))
else:
    print("No India test set (see the India cell above): india skipped")
TAGS = []
for tag, data in DATA:
    print(f"\n=== test split, {tag} ===")
    args = ("src/evaluate.py", "--arms", ARMS_YAML, "--data", data, "--split", "test", "--out-dir", "results/gen2",
            "--tag", tag, "--device", 0)
    if tag != "india":
        run(*args, watch=["results/gen2"])  # the watchdog restarts a stalled evaluation (it is deterministic)
        TAGS.append(tag)
        continue
    try:  # india is optional: a failure is reported and the other domains' results stand
        run(*args, watch=["results/gen2"])
        TAGS.append(tag)
    except Exception as e:
        print(f"\n*** WARNING: the india evaluation failed ({type(e).__name__}: {e}); its error is printed above. "
              f"india is left out of the tables; the other domains are not affected. ***")
        INDIA_NOTE = f"india: the evaluation failed ({type(e).__name__}, see the notebook log), left out"


In [ ]:
# The comparison reports written by evaluate.py
from IPython.display import Markdown, display
for tag in TAGS:
    display(Markdown(open(f"results/gen2/test_{tag}_comparison.md", encoding="utf-8").read()))


In [ ]:
# One table: test mAP@50 [95% bootstrap CI] per arm and domain; also saved as results/gen2/test_domains.md
import json
reports = {tag: json.load(open(f"results/gen2/test_{tag}_results.json")) for tag in TAGS}
header = "| Arm | " + " | ".join(f"{t} ({reports[t]['images']} images)" for t in TAGS) + " |"
lines = ["## Test mAP@50 [95% CI] per domain", "", header, "|---|" + "---|" * len(TAGS)]
for label, arm in arms.items():
    cells = []
    for t in TAGS:
        m = reports[t]["models"][label]
        lo, hi = m["bootstrap"]["map50_ci95"]
        cells.append(f"{m['metrics']['map50']:.3f} [{lo:.3f}, {hi:.3f}]")
    lines.append(f"| {label}: {arm['name']} | " + " | ".join(cells) + " |")
lines += ["", "Pairwise verdicts (paired bootstrap, 95%):"]
for t in TAGS:
    for p in reports[t]["pairwise"]:
        lines.append(f"- {t}, {p['b']} vs {p['a']}: {p['verdict']}")
if INDIA_NOTE:
    lines += ["", INDIA_NOTE]
table = "\n".join(lines) + "\n"
open("results/gen2/test_domains.md", "w", encoding="utf-8").write(table)
print(table)


### Files to download / where they go locally

From the **Output** tab of the finished version (all under `SwarmSort/results/gen2/`):

| File in the Output tab | Where it goes locally |
|---|---|
| `test_<tag>_results.json` (tags all, studio, real_world, india) | `results/gen2/` |
| `test_<tag>_comparison.md` | `results/gen2/` |
| `test_domains.md` (the combined arm x domain table) | `results/gen2/` |
| `build_report_india.json` (only if DWSD was built here: its build report and the MD5 cross-check) | `results/gen2/` |
| `plots/` (confusion matrices, per-class AP, samples, training curves) | `results/gen2/plots/` |

Commit them all. The copied weights, JSONs and the `runs/` folder are not needed (the JSONs and CSVs already came from
notebooks 1, 4 and 5).

### If the session dies

If a version stopped or froze, just run it again with the same inputs: the evaluation is deterministic (seeded
bootstrap), so it gives the same numbers, and nothing needs to be carried over. It still counts as the single use of
the test split.